# 02 — Analysis

Reads only what `01_collect.ipynb` produced. No network calls anywhere, and every
figure is drawn from `data/figures/*.parquet` rather than from the collectors — so
each figure's backing numbers can be inspected as a table and the notebook
re-runs offline.

**The discipline this notebook is held to**, from `PROJECT_BRIEF.md` §1 and the
module brief:

- Parameters are fixed **before** looking at results: NYT query forms in
  `collect/aliases.py`, Wikipedia title mappings and validity floors in
  `collect/wikipedia.py`, the normalization in `figures.build_relative_frame`,
  the collection priority in `watchlist.csv`.
- **Report the honest number.** A null result is a real result.
- **Mark underpowered results as such.**
- **No Tier B finding is promoted to a Tier A claim.**

### What changed since the first pass

Two free sources were added, and they reshaped the analysis:

- **Wikipedia pageviews** — dense, monthly, keyless, 2015→today. It covers the
  whole watchlist rather than the 12 companies NYT's quota allows, which roughly
  triples the usable sample.
- **The EDGAR event timeline** — in particular **DRS**, the *confidential* draft
  registration. It precedes the public S-1 by a median of ~96 days and by up to
  1,408, and it was secret when filed. That turns "does attention rise before
  the S-1?" into a sharper, testable question about information leakage.

In [ ]:
import json, sys
import numpy as np
import pandas as pd
sys.path.insert(0, "../..")

from research import figures
from research.collect import paths
from research.collect.edgar_enrich import read_watchlist_df
from research.collect.edgar_events import EVENTS_PARQUET
from research.collect.wikipedia import WIKI_PARQUET

pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 56)

print("figure frames:", figures.build_frames())
print(json.loads(figures.FIG_MANIFEST.read_text())["built_at"])

## 1 — How the Tier B watchlist differs from the Tier A population

**First figure, not an appendix.** Every Tier B number below is only
interpretable against it, because the watchlist was hand-assembled from
well-known names and is therefore biased by construction. The point of having a
census is to *show* the bias instead of disclaiming it.

In [ ]:
fig = figures.plot_cohort_comparison()

In [ ]:
cohort = pd.read_parquet(figures.FIG_COHORT)
for dim in cohort["dimension"].unique():
    sub = cohort[cohort["dimension"] == dim]
    print(f"\n--- {dim} ---")
    print(sub[["bucket", "tier_a", "tier_a_share", "tier_b", "tier_b_share"]]
          .to_string(index=False, float_format=lambda v: f"{v:.3f}"))

Read the deal-size panel first — it is the sharpest statement of the bias.
The watchlist concentrates in the largest bucket while the population
concentrates in the smallest, and it contains no company at all from the two
smallest buckets. A list of "companies everyone remembers going public" is close
to a list of the biggest offerings.

## 2 — The pre-listing event timeline

Before looking at any attention series, establish *when things happened*. The
public S-1 is not the start of the IPO process; the confidential DRS is.

In [ ]:
ev = pd.read_parquet(EVENTS_PARQUET)
gaps = ev["drs_to_s1_days"].dropna()
print(f"DRS present for {ev['drs_first'].notna().sum()} of {len(ev)} companies\n")
print("DRS -> public S-1 gap, in days:")
print(gaps.describe(percentiles=[.25, .5, .75, .9]).round(0).to_string())
print("\nThe median company was in confidential registration for about "
      f"{gaps.median()/30.44:.1f} months before the market could know.")

In [ ]:
display(ev.sort_values("drs_to_s1_days", ascending=False)[
    ["company", "drs_first", "s1_first", "drs_to_s1_days", "pricing_first",
     "s1_amendments", "form_d_count", "comment_rounds"]].head(14))

### Why this matters for the study design

A 24-month pre-S-1 window is supposed to capture "before anything happened". For
the companies below it does not even reach the confidential filing, so every
month in their window is *already inside* the registration process.

In [ ]:
outside = ev[ev["drs_to_s1_days"] > 730]
print(f"{len(outside)} of {len(ev)} companies have a DRS outside a 24-month "
      f"pre-S-1 window:")
display(outside[["company", "drs_first", "s1_first", "drs_to_s1_days"]])
print("\nFor these, 'attention before the S-1' is not a clean pre-event baseline.")

## 3 — Attention trajectory per company

Figure 1: monthly NYT articles, monthly Wikipedia views, daily price, on a shared
x-axis, with **three** vertical markers — confidential DRS, public S-1, listing.

The order of those markers is the point. Attention that rises between the first
two moved while the registration was still secret.

In [ ]:
panels = pd.read_parquet(figures.FIG_PANELS)
print("series available, by company count:")
print(panels[panels["value"].notna()].groupby("series")["company"]
      .nunique().to_string())
print("\nWikipedia covers the whole watchlist; NYT only the quota-limited 12.")

In [ ]:
# log_counts is one choice applied to every company or to none -- a per-company
# axis choice would make two panels look alike that are not. Left False.
have_both = sorted(set(panels.loc[panels["series"] == "nyt_articles", "company"])
                   & set(panels.loc[panels["series"] == "wikipedia_views", "company"]))
for company in have_both[:4]:
    figures.plot_company(company, log_counts=False)

## 4 — Does attention rise before the confidential filing, or after the public one?

The module's central question, now askable properly. Three windows, fixed before
looking, all measured **relative to the DRS** and expressed as **views per
month** so windows of different length are comparable:

- `baseline`  — months −24 to −13 before the DRS
- `confidential` — DRS month to the public S-1 month
- `public` — S-1 month to +3 months

Only companies with data in **all three** windows are used, so the comparison is
balanced rather than reflecting which companies happen to have long histories.

In [ ]:
wiki = pd.read_parquet(WIKI_PARQUET)
anchors = ev.set_index("company")[["drs_first", "s1_first"]].copy()
for c in anchors.columns:
    anchors[c] = pd.to_datetime(anchors[c], errors="coerce")

w = wiki[wiki["valid_attention"]].copy()
w["month_start"] = pd.to_datetime(w["month"] + "-01")
w = w.join(anchors, on="company").dropna(subset=["drs_first", "s1_first"])
w["m_drs"] = ((w["month_start"].dt.year - w["drs_first"].dt.year) * 12
              + (w["month_start"].dt.month - w["drs_first"].dt.month))
w["m_s1"] = ((w["month_start"].dt.year - w["s1_first"].dt.year) * 12
             + (w["month_start"].dt.month - w["s1_first"].dt.month))

def window_of(r):
    if -24 <= r["m_drs"] <= -13:
        return "baseline"
    if r["m_drs"] >= 0 and r["m_s1"] < 0:
        return "confidential"
    if 0 <= r["m_s1"] <= 3:
        return "public"
    return None

w["window"] = w.apply(window_of, axis=1)
per = (w.dropna(subset=["window"]).groupby(["company", "window"])["views"]
       .mean().unstack())
balanced = per.dropna()
print(f"{len(balanced)} of {per.shape[0]} companies have all three windows\n")
display(balanced[["baseline", "confidential", "public"]].round(0).astype(int))

In [ ]:
cols = ["baseline", "confidential", "public"]
print("median views per month across companies:")
print(balanced[cols].median().round(0).to_string())
print("\nratio to baseline (per company, then median):")
ratio = balanced[cols].div(balanced["baseline"], axis=0)
print(ratio[["confidential", "public"]].median().round(2).to_string())
print(f"\nn = {len(balanced)}. ", end="")
if len(balanced) >= 6:
    # Wilcoxon signed-rank without scipy: exact sign test on the paired
    # differences, which needs no distributional assumption and is honest at
    # this sample size.
    from math import comb
    for w1, w2 in (("baseline", "confidential"), ("confidential", "public")):
        d = (balanced[w2] - balanced[w1]).dropna()
        pos, n = int((d > 0).sum()), int((d != 0).sum())
        p = 2 * sum(comb(n, k) for k in range(pos, n + 1)) / 2 ** n if n else float("nan")
        print(f"\n  {w1} -> {w2}: {pos}/{n} companies rose, sign-test p = {min(p,1.0):.3f}")
print("\nRead the direction and the n. A sign test on a dozen-odd paired "
      "observations is weak evidence either way.")

### Read this carefully

If `confidential` is elevated over `baseline`, attention was already moving while
the filing was secret — consistent with leakage, but **also** consistent with the
company simply becoming more famous for ordinary reasons, which is very likely
what drives a company to file in the first place. This design cannot separate
those two, and the withdrawn-company comparison group is exactly what would
begin to: companies that filed confidentially and never listed. That group is not
collected yet.

If `public` is far above `confidential`, the S-1 itself is the attention event
and there is little sign of anticipation.

## 4b — X chatter before filing versus after listing

A second, independent instrument on the same question, from a narrow query:
`"<company>" IPO` over 90 days before the public S-1 and 90 days after the
listing. Equal-length windows, so the counts compare directly without
normalising.

Requiring the token `IPO` is what makes this affordable — the bare brand name is
a firehose of unrelated chatter, while this is the offering itself. It is also
the *right* query for a pre-filing window: a tweet saying "Figma IPO" before the
S-1 exists is anticipation.

**Read the `quality` column before the counts.** `advanced_search` has no
total-count field, so a count means paginating to exhaustion, and three
different things can happen:

| quality | meaning |
|---|---|
| `exact` | the provider ran out inside the page cap — a real count |
| `lower_bound` | cap hit while results were still in-window — a real floor |
| `unreliable_window` | cap hit but most results fell *outside* the window, i.e. the `since:`/`until:` operators were largely ignored — neither a count nor a bound |

The third category is not a rounding problem. Lyft's pre-filing window returned
**0 in-window tweets out of 160 fetched**; every one of the 13 unreliable windows
is a pre-filing window on an older listing. So this instrument degrades on
exactly the period the module most wants to measure, and those rows are dropped
rather than averaged in.

In [ ]:
from research.collect.twitter_windows import WINDOWS_PARQUET

tw = pd.read_parquet(WINDOWS_PARQUET)
print(f"{len(tw)} windows collected\n")
print(pd.crosstab(tw["window"], tw["quality"], margins=True).to_string())
print("\nEvery unreliable window is a pre-filing window:")
print(tw[tw["quality"] == "unreliable_window"]
      .groupby("window").size().to_string())
print("\nWorst offenders -- fraction of fetched tweets that fell outside the window:")
display(tw.nlargest(6, "off_target_fraction")[
    ["company", "window", "count", "out_of_window", "off_target_fraction", "quality"]])

### Counts are intervals, so compare them as intervals

Eight `pre_filing` windows were re-collected at a 20-page cap to convert lower
bounds into counts. Five converted. That helped, and it also introduced a trap
worth understanding, because the naive reading of the result is now wrong.

Deepening one side of a pair gives it more search effort than its partner.
Rivian's pre-filing window reads **360** against a post-listing **159** — but the
pre side got 20 pages and the post side 8, so the apparent inversion measures the
page budget, not the company. Three pairs invert that way.

The fix is not to compare page budgets but to treat every count as an interval:

| observation | interval |
|---|---|
| `exact` c | `[c, c]` |
| `lower_bound` c | `[c, ∞)` |

"post exceeds pre" is established **iff** `post_low > pre_high`, and vice versa.
This is looser than an equal-effort rule in one direction — Arm Holdings' pre
count is exactly 12 because it *exhausted* in 3 pages, so its partner's `≥160`
settles the direction regardless of effort. And it is stricter in another: it
refuses Klarna, where pre is exactly 209 and post is only known to be `≥158`, so
the true post value could sit on either side.

In [ ]:
pairs = pd.read_parquet(figures.FIG_WINDOWS)
print(f"{len(pairs)} usable pairs")
print(f"  direction established : {int(pairs['direction_established'].sum())}"
      f"  (post>pre {int(pairs['post_exceeds_pre'].sum())}, "
      f"pre>post {int(pairs['pre_exceeds_post'].sum())})")
print(f"  magnitude valid       : {int(pairs['magnitude_valid'].sum())}"
      f"  (both counts exact)\n")

print("=== ambiguous: no direction follows ===")
display(pairs.loc[~pairs["direction_established"],
                  ["company", "pre_filing", "pre_filing_q", "post_listing",
                   "post_listing_q", "incomparable_reason"]])

print("\n=== what deepening bought ===")
for c in ("Bumble", "Snowflake", "Peloton", "Birkenstock"):
    r = pairs.loc[pairs["company"] == c]
    if len(r):
        r = r.iloc[0]
        print(f"  {c:14} pre {r['pre_filing']:4.0f} exact vs post "
              f">={r['post_listing']:4.0f}  ->  established")
print("  Before deepening these four were censored on both sides and read as "
      "flat.\n  Four others (Klarna, Rivian, Bullish, Palantir) are now honestly "
      "ambiguous\n  rather than falsely flat -- also a gain.")

In [ ]:
# Only two-exact pairs support a magnitude.
exact = pairs[pairs["magnitude_valid"]].copy()
exact["ratio"] = np.where(exact["pre_filing"] > 0,
                          exact["post_listing"] / exact["pre_filing"], np.nan)
display(exact[["company", "pre_filing", "post_listing", "ratio"]]
        .sort_values("post_listing", ascending=False)
        .to_string(index=False, float_format=lambda v: f"{v:.1f}"))

from math import comb
d = exact["post_listing"] - exact["pre_filing"]
n, pos = int((d != 0).sum()), int((d > 0).sum())
if n:
    p = 2 * sum(comb(n, k) for k in range(pos, n + 1)) / 2 ** n
    print(f"\nsign test on the exact pairs: {pos}/{n} rose, p = {min(p, 1.0):.4f}")
med = exact.loc[exact["ratio"].notna(), "ratio"].median()
print(f"median ratio (non-zero baselines): {med:.1f}x")
print("\nFirefly Aerospace and Tempus AI had EXACTLY ZERO pre-filing posts "
      "matching\nthe query, then 101 and 44 after listing. No anticipation at "
      "all by this measure.")
print("\nAcross all 20 pairs where direction is established, every single one "
      "rises.\nNot one company had more pre-filing than post-listing chatter.")

In [ ]:
fig = figures.plot_windows()

### What this adds, and what it does not

It agrees with the Wikipedia result in §4 and by a much larger margin: the
listing is overwhelmingly the attention event, and pre-filing IPO-specific
chatter is close to absent for several companies. Two independent instruments
pointing the same way is worth more than either alone.

What it does **not** establish:

- **It is not a mention count.** It counts posts matching one narrow query. A
  company discussed constantly without the token "IPO" scores zero here, by
  design.
- **The censored pairs carry no magnitude**, only a direction, and 17 of 25
  pairs are censored on at least one side.
- **The pre-filing window is not a pre-event baseline.** It ends at the *public*
  S-1, and the median company filed confidentially ~96 days earlier — so most of
  this window sits inside the confidential registration period. ``
- **The direction was never in doubt.** That attention rises when a company
  starts trading is close to a tautology; the interesting question was the
  pre-filing run-up, and this instrument is weakest exactly there.

## 4c — Reddit, over the same two windows

A third instrument, and the one with the worst API. **`redditapis.com` has no
date-range parameter at all**: `q`, `sort`, and a `t` bucket
(`hour|day|week|month|year|all`) relative to *now*. `t=year` means "the last
twelve months", not "the year around Lyft's 2019 S-1".

So windows are bucketed **client-side**: sweep `t=all` sorted by `new`, page
backwards with `after`, and assign each post by its own `created_utc`. That was
hopeless for a broad query — one 100-post page of `q=Instacart` spanned **2.17
days**. With `"<brand>" IPO` a page spans **~600 days**, which makes it viable.
One sweep serves both windows, so this is cheaper per company than the X
collection.

Two corrections applied to the raw response:

- **A brand filter.** The provider matches loosely — 18 of 100 raw Lyft results
  never mention Lyft. Every post is checked against the brand and its aliases
  before counting; 11% of everything fetched was discarded.
- **Per-window completeness.** `sort=new` walks backwards from today, so the
  post-listing window is reached before the pre-filing one. A count is only a
  count if the sweep reached back past *that window's* start. Otherwise it is a
  floor — and a floor of zero means "never looked".

In [ ]:
from research.collect.reddit_windows import REDDIT_PARQUET

rd = pd.read_parquet(REDDIT_PARQUET)
print(f"{len(rd)} companies swept, {int(rd['pages'].sum())} billed calls\n")
print(f"both windows complete   : {int(rd['counts_complete'].sum())}")
print(f"direction established   : {int(rd['direction_established'].sum())}")
print(f"posts kept / discarded  : {int(rd['kept_posts'].sum())} / "
      f"{int(rd['discarded_no_mention'].sum())} "
      f"({100 * rd['discarded_no_mention'].sum() / (rd['kept_posts'].sum() + rd['discarded_no_mention'].sum()):.0f}% discarded)")

### Why 28 of 39 are incomplete — and why more budget would not help

This is the important cell. The incomplete sweeps did **not** run out of my page
budget; Reddit stopped issuing a pagination cursor. Its own response says a
platform cut-off and a genuine end-of-data are indistinguishable from outside,
so the sweep simply cannot go further at any price.

In [ ]:
inc = rd[~rd["counts_complete"]]
print("pages used by the incomplete sweeps (my cap was 7):")
print(inc["pages"].value_counts().sort_index().to_string())
print(f"\ncursor exhausted before the window was reached: "
      f"{int(inc['listing_ended'].sum())} of {len(inc)}")
print("\nNone approached the 7-page cap, so the limit is Reddit's listing "
      "cut-off,\nnot the call budget. 169 of 270 calls remain unspent because "
      "spending them\ncould not change any of these rows.")
print("\nWhich companies are lost, and they are the busy recent ones:")
display(inc[["company", "listing_date", "pages", "oldest_seen"]]
        .sort_values("listing_date", ascending=False).head(10))

Note the failure mode is the **mirror image** of the X windows. There, all 13
unreliable windows were *pre-filing* windows on *older* listings. Here, the
losses are *recent, heavily-discussed* companies, because `sort=new` starts at
today and a busy query burns its cursor before reaching back. Two scrapers, two
opposite blind spots — which is a reason to report them separately rather than
pooling them into one "social" number.

In [ ]:
complete = rd[rd["counts_complete"]].sort_values("post_listing_count",
                                                 ascending=False)
display(complete[["company", "s1_date", "listing_date", "pre_filing_count",
                  "post_listing_count", "pages", "oldest_seen"]]
        .to_string(index=False))

from math import comb
d = complete["post_listing_count"] - complete["pre_filing_count"]
n, pos = int((d != 0).sum()), int((d > 0).sum())
p_val = 2 * sum(comb(n, k) for k in range(pos, n + 1)) / 2 ** n
print(f"\nrose after listing: {int((d > 0).sum())} of {len(complete)}")
print(f"sign test: {pos}/{n}, p = {min(p_val, 1.0):.4f}")
zeros = complete[complete["pre_filing_count"] == 0]["company"].tolist()
print(f"\nExactly zero pre-filing Reddit posts (and complete, so a real zero): "
      f"{', '.join(zeros)}")

In [ ]:
fig = figures.plot_windows("reddit")

### Three instruments, one direction

| instrument | usable | rose | test |
|---|---|---|---|
| Wikipedia (DRS-anchored) | 20 companies | 18/20 into the public window | p < 0.001 |
| X windows | 20 pairs, direction established | **20/20** | 8/8 exact, p = 0.008 |
| Reddit windows | 11 complete pairs | **11/11** | p = 0.001 |

Three sources with different corpora, different failure modes, and different
operators all say the listing is the attention event. That convergence is worth
more than any one of them.

It is also, honestly, the least surprising possible result — attention rising
when a company starts trading is close to tautological. The question with real
content was the *pre-filing run-up*, and every instrument here is weakest exactly
there: Wikipedia's confidential-window rise is not significant (p = 0.115), the X
pre-filing windows are where all 13 unreliable rows sit, and Reddit's losses are
concentrated in the companies with the most pre-filing chatter.

What would actually settle it is the withdrawn-company comparison group —
companies that filed confidentially and never listed. That is still not
collected, and it remains the highest-value work outstanding.

## 5 — Cohort view in relative time, on all three anchors

Figure 2. Calendar-time overlay would mostly show that 2021 and 2025 were
different markets. Relative time is the only legitimate form of cross-company
comparison here.

Counts are normalized to each company's own window total before overlay, fixed
before looking — raw counts would make SpaceX and Klaviyo incomparable.

In [ ]:
for anchor in ("drs", "s1", "listing"):
    figures.plot_relative_time("wikipedia_views", anchor=anchor)

In [ ]:
rel = pd.read_parquet(figures.FIG_RELATIVE)
wv = rel[(rel["series"] == "wikipedia_views") & rel["share"].notna()]
print(f"companies in the Wikipedia overlay: {wv['company'].nunique()}")
nyt_rel = rel[(rel["series"] == "nyt_articles") & rel["share"].notna()]
print(f"companies in the NYT overlay:       {nyt_rel['company'].nunique()}")
print("\nThe Wikipedia series is the one with enough companies to say anything.")

The withdrawn-company group belongs on these axes as a second series,
anchored on filing date since those companies never list. The 431 `withdrawn`
rows are in the Tier A census and are **not yet collected against**, so
survivorship in Tier B is total and no claim about what distinguishes a company
that lists from one that does not is supported. See README limitation 2.

## 6 — Attention around listing versus subsequent return

Return is measured from the **opening print**, as `PROJECT_BRIEF.md` §2
specifies, at 30/60/90 trading sessions.

Read the `n` before the correlation.

In [ ]:
post = pd.read_parquet(figures.FIG_POST)
rows = []
for company, g in post.groupby("company"):
    g = g.sort_values("session")
    opening = g["close"].iloc[0]
    rec = {"company": company, "sessions": int(g["session"].max()) + 1}
    for h in (30, 60, 90):
        rec[f"ret_{h}d"] = (g.loc[g["session"] == h, "close"].iloc[0] / opening - 1
                            if (g["session"] == h).any() else np.nan)
    rows.append(rec)
returns = pd.DataFrame(rows).set_index("company")

# Attention in the listing month, from the dense series so all 12 have a value.
listing_month = post.groupby("company")["month"].first()
wm = wiki[wiki["valid_attention"]].set_index(["company", "month"])["views"]
returns["wiki_listing_month"] = [
    wm.get((c, listing_month.get(c)), np.nan) for c in returns.index]
display(returns.round(3))
print("\nNaN return = not yet traded that many sessions. Nothing is imputed.")

In [ ]:
usable = returns.dropna(subset=["wiki_listing_month"])
for h in (30, 60, 90):
    sub = usable.dropna(subset=[f"ret_{h}d"])
    n = len(sub)
    if n < 4:
        print(f"{h}d: n={n} -- too few to correlate.")
        continue
    r = sub["wiki_listing_month"].rank().corr(sub[f"ret_{h}d"].rank())
    z, se = np.arctanh(r), 1 / np.sqrt(n - 3)
    lo, hi = np.tanh(z - 1.96 * se), np.tanh(z + 1.96 * se)
    flag = "  <- interval spans 0" if lo < 0 < hi else ""
    print(f"{h}d: n={n:2d}  Spearman rho={r:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]{flag}")
print("\nSpearman on ranks, because both variables are heavily skewed.")
print("Any interval spanning zero is a NULL RESULT at this sample size -- the "
      "expected and publishable outcome. See PROJECT_BRIEF.md section 2.")

### Before banking the 30-day number

At the time of writing the 30-session correlation comes out around **−0.66 with a
95% interval that excludes zero**, and its sign is the Popularity Trap direction:
more attention, worse subsequent return. It would be very easy to write that up.
It should not be.

Three reasons, all visible above:

1. **n ≈ 10.** At that size the interval is enormous even when it excludes zero,
   and a single company moving ranks can flip it.
2. **Three horizons were tested and only one is nominally significant.** With
   three correlated tests at α=0.05, one hit is close to what chance produces.
   No multiple-comparison correction is applied below, and after even the
   crudest one it does not survive.
3. **The sign is unstable across horizons** — 30d negative, 60d *positive*, 90d
   negative. A real effect on price drift does not reverse and reverse again over
   two months; sampling noise does exactly that.

The cell below states this arithmetically rather than leaving it to judgement.

In [ ]:
# How ordinary is "one significant result out of three"?
alpha, k = 0.05, 3
print(f"P(at least one of {k} independent tests significant at {alpha}) = "
      f"{1 - (1 - alpha) ** k:.3f}")
print(f"Bonferroni-corrected threshold for {k} tests: {alpha / k:.4f}\n")

signs = {}
for h in (30, 60, 90):
    sub = usable.dropna(subset=[f"ret_{h}d"])
    if len(sub) >= 4:
        r = sub["wiki_listing_month"].rank().corr(sub[f"ret_{h}d"].rank())
        signs[h] = r
print("sign of rho by horizon:",
      {h: ("+" if v > 0 else "-") for h, v in signs.items()})
print("\nVERDICT: the horizons disagree in sign and only one of three clears an "
      "uncorrected threshold at n~10. This is a NULL RESULT. Reporting the 30d "
      "figure as the Popularity Trap would be exactly the error "
      "PROJECT_BRIEF.md section 1 exists to prevent.")

In [ ]:
# What effect size this design could even detect.
print("smallest |r| reaching p<0.05, by sample size:")
for n, tc in ((12, 2.228), (25, 2.069), (40, 2.024), (150, 1.976)):
    print(f"  n={n:3d}  need |r| >= {tc/np.sqrt(n-2+tc**2):.2f}")
print("\nAttention-return effects in the literature are ~0.1-0.3. At n=12 this "
      "design cannot see them, which is a property of the sample, not the sensors.")

## 7 — Post-listing overlay

Figure 3. The post-listing tail is the only region where price and attention
coexist, so the only place a single-panel overlay is defensible.

Marker size encodes **volume only**. No sentiment scoring is in scope and no
sentiment proxy is derived from counts.

In [ ]:
fig = figures.plot_post_listing(sorted(returns.index)[:6])

## 8 — Do the attention sources agree?

NYT and Wikipedia exist for the same company-months, so this is checkable.

Prior, stated before looking: they measure **different things** — NYT is elite
press attention, Wikipedia is public curiosity. Moderate correlation is the
expected result. Strong disagreement is informative rather than a failure, and
where they diverge Wikipedia is the denser instrument while NYT is the more
editorially filtered one.

In [ ]:
monthly = panels[panels["resolution"] == "monthly"]
wide = monthly.pivot_table(index=["company", "x"], columns="series",
                           values="value", aggfunc="first").reset_index()
both = wide.dropna(subset=["nyt_articles", "wikipedia_views"])
print(f"company-months with both series: {len(both)}")
if len(both) >= 10:
    rho = both["nyt_articles"].rank().corr(both["wikipedia_views"].rank())
    print(f"Spearman rho = {rho:+.3f} pooled across companies, n = {len(both)}\n")
    per_co = (both.groupby("company")
              .apply(lambda g: g["nyt_articles"].rank()
                     .corr(g["wikipedia_views"].rank()), include_groups=False)
              .dropna().sort_values())
    print("within-company Spearman rho:")
    print(per_co.round(3).to_string())
    print("\nPooling across companies inflates the correlation, because big "
          "companies score high on both. The within-company figures are the "
          "honest ones.")

## 9 — What this notebook does and does not support

Fill this in against the numbers actually printed above.

**Supported as written**

- The Tier A census (§1) is a full-population measurement; its counts stand alone
  and reproduce offline.
- The composition comparison (§1) is a measured statement about how unlike the
  population the watchlist is.
- The event timeline (§2) is a measured fact about filing behaviour, from primary
  sources, for every company that filed: confidential registration precedes
  public registration by a median of ~96 days.
- Per-company trajectories (§3) describe those companies.

**Not supported, and not to be written up as if it were**

- Any claim about "IPOs" in general drawn from the watchlist.
- Any attention→return relationship at this sample size, in either direction.
- **Leakage as a causal claim.** Elevated attention during the confidential
  window is equally consistent with a company becoming more famous for ordinary
  reasons — which is plausibly *why* it filed. Separating those needs the
  withdrawn-company group.
- Any statement about what separates a company that lists from one that
  withdraws — that group is not collected.
- Any Reddit, GNews, or X series. The first two could not deliver the window and
  were dropped; the third has no data because the account's credits are spent.